In [1]:
from pathlib import Path
import re
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter


# ============================================================
# 1. LOCATE POLICY DOCUMENTS
# ============================================================

POLICY_DIR = Path("../data/company_policy")

if not POLICY_DIR.exists():
    raise FileNotFoundError(f"Policy directory not found: {POLICY_DIR}")

policy_files = sorted(POLICY_DIR.glob("*.txt"))

print(f"Found {len(policy_files)} policy documents:\n")

for file in policy_files:
    print(file.name)


# ============================================================
# 2. INGEST DOCUMENTS
# ============================================================

documents = []

for file_path in policy_files:

    try:
        content = file_path.read_text(encoding="utf-8")

        # Handle empty documents
        if not content.strip():
            print(f"Skipping empty document: {file_path.name}")
            continue

        documents.append({
            "name": file_path.name,
            "content": content
        })

    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")


print(f"\nSuccessfully loaded {len(documents)} documents.")


# ============================================================
# 3. DOCUMENT STATISTICS
# ============================================================

document_stats = []

for doc in documents:

    document_stats.append({
        "document": doc["name"],
        "characters": len(doc["content"]),
        "words": len(doc["content"].split()),
        "lines": len(doc["content"].splitlines())
    })

stats_df = pd.DataFrame(document_stats)

print("\nDocument Statistics:")
display(stats_df)


# ============================================================
# 4. CLEAN DOCUMENTS
# ============================================================

def clean_text(text):

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove unnecessary spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove leading/trailing whitespace from each line
    lines = [line.strip() for line in text.splitlines()]

    # Reconstruct cleaned text
    text = "\n".join(lines).strip()

    return text


cleaned_documents = []

for doc in documents:

    cleaned_content = clean_text(doc["content"])

    if not cleaned_content:
        print(f"Skipping empty document after cleaning: {doc['name']}")
        continue

    cleaned_documents.append({
        "name": doc["name"],
        "content": cleaned_content
    })


print(f"\nCleaned {len(cleaned_documents)} documents.")


# ============================================================
# 5. CREATE METADATA
# ============================================================

def get_metadata(filename):

    if "travel_policy_india" in filename:
        return {
            "source": filename,
            "policy_type": "travel",
            "country": "India"
        }

    elif "travel_policy_us" in filename:
        return {
            "source": filename,
            "policy_type": "travel",
            "country": "US"
        }

    elif "airport_policy" in filename:
        return {
            "source": filename,
            "policy_type": "airport",
            "country": "Global"
        }

    elif "employee_eligibility" in filename:
        return {
            "source": filename,
            "policy_type": "eligibility",
            "country": "Global"
        }

    elif "expense_policy" in filename:
        return {
            "source": filename,
            "policy_type": "expense",
            "country": "Global"
        }

    elif "cancellation_policy" in filename:
        return {
            "source": filename,
            "policy_type": "cancellation",
            "country": "Global"
        }

    elif "approval_policy" in filename:
        return {
            "source": filename,
            "policy_type": "approval",
            "country": "Global"
        }

    else:
        return {
            "source": filename,
            "policy_type": "unknown",
            "country": "unknown"
        }


documents_with_metadata = []

for doc in cleaned_documents:

    metadata = get_metadata(doc["name"])

    documents_with_metadata.append({
        "text": doc["content"],
        "metadata": metadata
    })


print("\nMetadata:")
for doc in documents_with_metadata:
    print(doc["metadata"])


# ============================================================
# 6. CHUNK THE DOCUMENTS
# ============================================================

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)


# ============================================================
# 7. CREATE FINAL CHUNKS
# ============================================================

chunks = []

for doc in documents_with_metadata:

    split_texts = text_splitter.split_text(doc["text"])

    for i, chunk_text in enumerate(split_texts):

        chunks.append({
            "text": chunk_text,
            "metadata": {
                **doc["metadata"],
                "chunk_id": i
            }
        })


print(f"\nTotal chunks created: {len(chunks)}")


# ============================================================
# INSPECT FIRST 5 CHUNKS
# ============================================================

print("\nSample Chunks:\n")

for i, chunk in enumerate(chunks[:5]):

    print("=" * 80)
    print(f"Chunk {i}")
    print("Metadata:", chunk["metadata"])
    print("\nText:")
    print(chunk["text"])

Found 7 policy documents:

airport_policy.txt
approval_policy.txt
cancellation_policy.txt
employee_eligibility.txt
expense_policy.txt
travel_policy_india.txt
travel_policy_us.txt

Successfully loaded 7 documents.

Document Statistics:


,document,characters,words,lines
0,airport_policy.txt,1427,221,40
1,approval_policy.txt,1378,207,38
2,cancellation_policy.txt,1353,204,37
3,employee_eligibility.txt,1518,215,56
4,expense_policy.txt,1233,195,39
5,travel_policy_india.txt,1642,248,35
6,travel_policy_us.txt,1571,229,35



Cleaned 7 documents.

Metadata:
{'source': 'airport_policy.txt', 'policy_type': 'airport', 'country': 'Global'}
{'source': 'approval_policy.txt', 'policy_type': 'approval', 'country': 'Global'}
{'source': 'cancellation_policy.txt', 'policy_type': 'cancellation', 'country': 'Global'}
{'source': 'employee_eligibility.txt', 'policy_type': 'eligibility', 'country': 'Global'}
{'source': 'expense_policy.txt', 'policy_type': 'expense', 'country': 'Global'}
{'source': 'travel_policy_india.txt', 'policy_type': 'travel', 'country': 'India'}
{'source': 'travel_policy_us.txt', 'policy_type': 'travel', 'country': 'US'}

Total chunks created: 28

Sample Chunks:

Chunk 0
Metadata: {'source': 'airport_policy.txt', 'policy_type': 'airport', 'country': 'Global', 'chunk_id': 0}

Text:
UBER FOR BUSINESS - SAMPLE AIRPORT TRAVEL POLICY
Policy Version: 1.0
Effective Date: 2026-01-01
Policy Owner: Airport Travel Program

1. Purpose
This sample policy defines rules for company-sponsored Uber trips to and from

In [4]:
print("total chunks:",len(chunks))
print("Chunks per document")
for source , count in chunk_stats.groupby("source").size().items():
    print(f"{source}:{count}")

total chunks: 28
Chunks per document
airport_policy.txt:4
approval_policy.txt:3
cancellation_policy.txt:4
employee_eligibility.txt:4
expense_policy.txt:3
travel_policy_india.txt:5
travel_policy_us.txt:5


In [3]:
# Create chunk statistics

chunk_stats = pd.DataFrame([
    {
        "chunk_id": i,
        "source": chunk["metadata"]["source"],
        "policy_type": chunk["metadata"]["policy_type"],
        "country": chunk["metadata"]["country"],
        "characters": len(chunk["text"]),
        "words": len(chunk["text"].split())
    }
    for i, chunk in enumerate(chunks)
])

print("Total chunks:", len(chunks))

print("\nChunks per document:")
print(
    chunk_stats
    .groupby("source")
    .size()
    .reset_index(name="chunk_count")
)

Total chunks: 28

Chunks per document:
                     source  chunk_count
0        airport_policy.txt            4
1       approval_policy.txt            3
2   cancellation_policy.txt            4
3  employee_eligibility.txt            4
4        expense_policy.txt            3
5   travel_policy_india.txt            5
6      travel_policy_us.txt            5


In [ ]:
print(chunks[0])

In [5]:
experiments = [
    (300, 50),
    (500, 100),
    (800, 150)
]

results = []

for chunk_size, overlap in experiments:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    total_chunks = 0

    for doc in documents_with_metadata:
        total_chunks += len(splitter.split_text(doc["text"]))

    results.append({
        "chunk_size": chunk_size,
        "chunk_overlap": overlap,
        "total_chunks": total_chunks
    })

experiment_df = pd.DataFrame(results)

experiment_df

,chunk_size,chunk_overlap,total_chunks
0,300,50,49
1,500,100,28
2,800,150,18


# Day 1 Documentation

## Objective

Prepare the fictional company travel policy documents for use in
an AI-powered Travel & Policy Assistant.

## Document Processing

Seven policy documents were loaded from `data/company_policy/`.

The documents were:
- Ingested using Python
- Checked for missing/empty content
- Cleaned by normalizing whitespace and formatting
- Enriched with metadata
- Split into smaller chunks

## Metadata

Each chunk contains:

- `source` - original policy document
- `policy_type` - type of policy
- `country` - applicable country/region
- `chunk_id` - chunk identifier

## Chunking

Three configurations were experimented with:

- 300 characters / 50 overlap
- 500 characters / 100 overlap
- 800 characters / 150 overlap

The initial configuration selected for the project is:

**500 characters with 100 characters overlap**

The smaller configuration can produce highly focused chunks but may
lose context. Larger chunks preserve more context but may contain
unrelated information.

A recursive splitting strategy was selected so that the splitter
attempts to preserve meaningful boundaries such as paragraphs and
lines.

## Prompt Engineering

The following prompting approaches were designed:

1. P.T.C.F. prompting
2. Role-based prompting
3. Few-shot prompting
4. Structured output prompting

These prompts will be used later when integrating the LLM.

## Day 1 Result

The final preprocessing pipeline is:

Documents
→ Ingestion
→ Cleaning
→ Metadata
→ Chunking
→ Prompt Design

The resulting chunks are ready for Day 2 embedding and vector-search
processing.